# Classification with PyTorch

This chapter builds classifiers from first principles and studies **logits, CrossEntropyLoss, validation, model capacity, and nonlinear decision boundaries**.

The experiments progress from a simple three-class problem to the Two Moons dataset, where a linear model fails and a nonlinear network succeeds.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_moons

torch.manual_seed(42)

## 1. A simple three-class dataset

Each example has two features and an integer class label:

```text
x = [feature_1, feature_2]
y ∈ {0, 1, 2}
```

In [ ]:
class_0 = torch.randn(100, 2) + torch.tensor([-3.0, 0.0])
class_1 = torch.randn(100, 2) + torch.tensor([3.0, 0.0])
class_2 = torch.randn(100, 2) + torch.tensor([0.0, 3.0])

X = torch.cat([class_0, class_1, class_2], dim=0)
y = torch.cat([
    torch.zeros(100),
    torch.ones(100),
    torch.full((100,), 2)
]).long()

print(X.shape, y.shape)

## 2. Classifier architecture

```text
2 features
    ↓
Linear(2 → 16)
    ↓
ReLU
    ↓
Linear(16 → 3)
    ↓
3 logits
```

The final layer produces **raw logits**, not probabilities. `CrossEntropyLoss` consumes these logits directly, so no Softmax is added during training.

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 16)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(16, 3)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        return self.layer2(x)

model = Classifier()
print(model)

In [ ]:
sample = X[0].unsqueeze(0)

logits = model(sample)
predicted_class = logits.argmax(dim=1)

loss_fn = nn.CrossEntropyLoss()
sample_loss = loss_fn(logits, y[0].unsqueeze(0))

print("Input:", sample)
print("Logits:", logits)
print("Predicted class:", predicted_class)
print("Actual class:", y[0])
print("Loss:", sample_loss.item())

## 3. Train/validation split

Training data updates parameters. Validation data is held out from those updates.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=32,
    shuffle=False
)

In [ ]:
model = Classifier()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

epochs = 100

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0

    for batch_X, batch_y in train_loader:
        prediction = model(batch_X)
        loss = loss_fn(prediction, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        predicted_classes = prediction.argmax(dim=1)
        correct += (predicted_classes == batch_y).sum().item()
        total += batch_y.size(0)

    if epoch % 10 == 0:
        print(f"Epoch: {epoch:3d} | Loss: {epoch_loss / len(train_loader):.6f} | Accuracy: {correct / total:.2%}")

In [ ]:
model.eval()
val_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch_X, batch_y in val_loader:
        prediction = model(batch_X)
        loss = loss_fn(prediction, batch_y)
        val_loss += loss.item()
        predicted_classes = prediction.argmax(dim=1)
        correct += (predicted_classes == batch_y).sum().item()
        total += batch_y.size(0)

print(f"Validation Loss: {val_loss / len(val_loader):.6f}")
print(f"Validation Accuracy: {correct / total:.2%}")

## 4. Making the problem harder

Increasing cluster spread creates more overlap. In repeated experiments, this reduced validation accuracy and made model capacity visible.

Observed mean validation accuracies from five runs:

| Architecture | Mean validation accuracy |
|---|---:|
| `2 → 16 → 3` | 76.00% |
| `2 → 32 → 3` | 82.00% |
| `2 → 64 → 3` | 80.67% |
| `2 → 16 → 16 → 3` | 79.33% |
| `2 → 32 → 32 → 3` | 81.33% |

These are toy experiments, not universal benchmarks. More capacity can help, but more neurons or layers do not automatically improve generalization.

## 5. Visualizing decision boundaries

With two input features, we can evaluate the model on a dense grid and visualize which class it predicts at each point.

In [ ]:
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    model.eval()
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    x1 = torch.linspace(x_min, x_max, 300)
    x2 = torch.linspace(y_min, y_max, 300)
    grid_x1, grid_x2 = torch.meshgrid(x1, x2, indexing="ij")
    grid = torch.stack([grid_x1.flatten(), grid_x2.flatten()], dim=1)
    with torch.no_grad():
        predictions = model(grid).argmax(dim=1)
    Z = predictions.reshape(grid_x1.shape)
    plt.contourf(grid_x1.numpy(), grid_x2.numpy(), Z.numpy(), alpha=0.3)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors="black")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(title)
    plt.show()

plot_decision_boundary(model, X, y)

## 6. Nonlinear classification: Two Moons

The Two Moons dataset contains two curved classes that wrap around each other. A single linear layer cannot create the curved boundary required to separate them.

In [ ]:
X_moons, y_moons = make_moons(n_samples=1000, noise=0.15, random_state=42)
X_moons = torch.tensor(X_moons, dtype=torch.float32)
y_moons = torch.tensor(y_moons, dtype=torch.long)

plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Two Moons Dataset")
plt.show()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_moons, y_moons, test_size=0.2, random_state=42)
moon_train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
moon_val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32, shuffle=False)

### Linear baseline

```text
2 → 2
```

A single linear layer can only learn a linear decision boundary.

In [ ]:
class LinearClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(2, 2)

    def forward(self, x):
        return self.layer(x)

linear_model = LinearClassifier()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(linear_model.parameters(), lr=0.01)

for epoch in range(200):
    linear_model.train()
    for batch_X, batch_y in moon_train_loader:
        prediction = linear_model(batch_X)
        loss = loss_fn(prediction, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

linear_model.eval()
correct = total = 0
with torch.no_grad():
    for batch_X, batch_y in moon_val_loader:
        prediction = linear_model(batch_X)
        predicted_classes = prediction.argmax(dim=1)
        correct += (predicted_classes == batch_y).sum().item()
        total += batch_y.size(0)

print(f"Linear validation accuracy: {correct / total:.2%}")
plot_decision_boundary(linear_model, X_moons, y_moons, "Linear Classifier — Two Moons")

### Nonlinear model

Adding hidden layers and ReLU changes the function the network can represent:

```text
2 → 32 → 32 → 2
      ↓       ↓
     ReLU    ReLU
```

The nonlinear activations allow the network to learn nonlinear decision boundaries.

In [ ]:
class MoonClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 32)
        self.layer2 = nn.Linear(32, 32)
        self.layer3 = nn.Linear(32, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.layer3(x)

moon_model = MoonClassifier()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(moon_model.parameters(), lr=0.01)

for epoch in range(500):
    moon_model.train()
    for batch_X, batch_y in moon_train_loader:
        prediction = moon_model(batch_X)
        loss = loss_fn(prediction, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

moon_model.eval()
correct = total = 0
with torch.no_grad():
    for batch_X, batch_y in moon_val_loader:
        prediction = moon_model(batch_X)
        predicted_classes = prediction.argmax(dim=1)
        correct += (predicted_classes == batch_y).sum().item()
        total += batch_y.size(0)

print(f"Nonlinear validation accuracy: {correct / total:.2%}")
plot_decision_boundary(moon_model, X_moons, y_moons, "Nonlinear Classifier — Two Moons")

## Key conclusions

- **Classification:** maps inputs to discrete classes; `CrossEntropyLoss` expects integer class indices.
- **Logits:** raw class scores; `argmax` selects the highest-scoring class.
- **Capacity:** width and depth increase representational capacity, but more capacity does not guarantee better validation performance.
- **Nonlinearity:** without nonlinear activations, a stack of linear layers is still equivalent to one linear transformation. ReLU allows multi-layer networks to represent nonlinear functions.
- **Validation:** held-out data gives a better estimate of generalization than training accuracy.

```text
Input
  ↓
Linear transformations
  ↓
Nonlinear activations
  ↓
Learned representation
  ↓
Class logits
  ↓
CrossEntropyLoss
  ↓
Backpropagation
  ↓
Parameter updates
```